# Notebook B — BiAB-IoT Attack Sweep + Ablation (Tables 3 and 4)

Reproduces:

* **Table 3** — accuracy under label-flipping poisoning at 0 / 5 / 10 / 15 / 20 %
  malicious clients, for **standard FedAvg** (as a baseline reference — B2) and
  for the **full BiAB-IoT** (P1 + P2).
* **Table 4** — ablation showing what each mechanism contributes:
    * *P1 only* — reputation-triggered exclusion, unweighted aggregation of
      the retained clients.
    * *P2 only* — reputation-weighted aggregation, no exclusion.
    * *P1 + P2* — the full BiAB-IoT: exclusion of detected attackers and
      reputation-weighted aggregation of the remaining trusted clients.

Detection at each gateway is done via **Isolation Forest anomaly scoring**
(matching Algorithm 2 in the manuscript). A dual condition is required to
flag a client for exclusion: the client's local anomaly score must exceed
`tau_anomaly` **and** its current reputation must be below `tau_reputation`.

**Prerequisites.** `binary_dataset.pkl` on Drive and `biab_common.py` under
`/content/drive/MyDrive/path1_code/`. Notebook A does not need to have been
run first — this notebook is self-contained.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install --upgrade scikit-learn tensorflow
import os, sys, random
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/path1_code')
from biab_common import (
    load_dataset, make_clients, poison_clients, make_model, train_local,
    fed_average, evaluate, save_result, seed_everything, SEED, RESULT_DIR,
)
seed_everything(SEED)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
data = load_dataset()
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
input_dim = X_train.shape[1]

NUM_CLIENTS = 500
CLIENTS_PER_ROUND = 50
ROUNDS = 10

# Manuscript defaults (Simulation Configuration section)
TAU_ANOMALY    = 0.15   # anomaly-score threshold (Algorithm 2)
TAU_REPUTATION = 0.40   # reputation threshold   (Algorithm 2)
DELTA_PENALTY  = 0.30   # reputation drop per confirmed anomaly
GAMMA_RECOVERY = 0.01   # reputation gain per clean round
R_MIN          = 0.30   # aggregation gate

print('config: TAU_ANOMALY=%.2f, TAU_REPUTATION=%.2f, DELTA=%.2f, GAMMA=%.2f, R_MIN=%.2f'
      % (TAU_ANOMALY, TAU_REPUTATION, DELTA_PENALTY, GAMMA_RECOVERY, R_MIN))

## Isolation-Forest anomaly score per client

`Algorithm 2` in the manuscript computes a normalised anomaly score in `[0, 1]`
using an Isolation Forest trained on the gateway's telemetry. Here we implement
that literally: for each round, at each gateway, fit an IF on the pooled
client data, score every client's data, and normalise to `[0, 1]`.

In [ ]:
from sklearn.ensemble import IsolationForest

def compute_anomaly_scores(pool_clients):
    """Return one normalised anomaly score per client in `pool_clients`.
    Higher = more anomalous."""
    all_X = np.vstack([Xc[:2000] for Xc, _ in pool_clients])
    m = IsolationForest(n_estimators=50, contamination=0.18,
                         random_state=SEED, n_jobs=-1)
    m.fit(all_X)
    scores = []
    for Xc, _ in pool_clients:
        # decision_function: higher = more normal. Invert and normalise.
        s = -np.mean(m.decision_function(Xc[:2000]))
        scores.append(s)
    s = np.array(scores)
    if s.max() - s.min() < 1e-9:
        return np.zeros_like(s)
    return (s - s.min()) / (s.max() - s.min())

## Reputation update rule

We use the manuscript's Section 5 reputation-update function:

* On confirmed anomaly (dual condition triggers): `R -> max(0, R - delta)`.
* On a clean round with no trigger: `R -> min(1, R + gamma)`.

`gamma = 0.01`, `delta = 0.3`. A single penalty therefore requires 30
consecutive clean rounds to recover — the asymmetric structure the paper
attributes to BiAB-IoT.

In [ ]:
def run_biab_iot(attack_pct: float, use_P1: bool, use_P2: bool,
                 rounds: int = ROUNDS, seed: int = SEED):
    """Single BiAB-IoT run with configurable P1 and P2 mechanisms.

    - P1 (AI->Blockchain policy trigger): when a client's IF anomaly score
      exceeds tau_anomaly AND its reputation is below tau_reputation, apply
      the delta-penalty and exclude it from this round's aggregation.
      When P1 is off, no exclusion is done.

    - P2 (Blockchain->AI reputation-weighted aggregation): if enabled,
      aggregation uses reputation as the weight. When P2 is off, aggregation
      is unweighted (equal weights over the retained clients).
    """
    seed_everything(seed)
    clients = make_clients(X_train, y_train, NUM_CLIENTS, seed)
    poisoned, mal_ids = poison_clients(clients, attack_pct, seed)
    reputation = np.ones(NUM_CLIENTS, dtype=np.float64)

    global_model = make_model(input_dim)
    global_weights = global_model.get_weights()

    for rnd in range(rounds):
        pool_ids = random.sample(range(NUM_CLIENTS), CLIENTS_PER_ROUND)
        pool = [poisoned[cid] for cid in pool_ids]

        # ---- Anomaly scoring ---------------------------------------------
        anom = compute_anomaly_scores(pool)

        # ---- P1 dual-condition trigger -----------------------------------
        keep_idx = []
        for j, cid in enumerate(pool_ids):
            triggered = (anom[j] > TAU_ANOMALY) and (reputation[cid] < TAU_REPUTATION) \
                        if use_P1 else False
            if triggered:
                reputation[cid] = max(0.0, reputation[cid] - DELTA_PENALTY)
            else:
                # slow recovery for good behaviour
                reputation[cid] = min(1.0, reputation[cid] + GAMMA_RECOVERY)
                keep_idx.append(j)

        # If P1 is disabled we keep everyone
        if not use_P1:
            keep_idx = list(range(len(pool_ids)))

        # Aggregation gate: keep only if reputation >= R_MIN
        keep_idx = [j for j in keep_idx if reputation[pool_ids[j]] >= R_MIN]
        if not keep_idx:
            print(f'  round {rnd+1}: no clients retained, skipping aggregation')
            continue

        # ---- Local training ---------------------------------------------
        local_ws = []
        for j in keep_idx:
            Xc, yc = pool[j]
            local_ws.append(train_local(Xc, yc, global_weights, input_dim))

        # ---- P2 reputation-weighted aggregation --------------------------
        if use_P2:
            w_scale = [reputation[pool_ids[j]] for j in keep_idx]
        else:
            w_scale = None

        global_weights = fed_average(local_ws, weights_scale=w_scale)
        global_model.set_weights(global_weights)

        n_excluded = CLIENTS_PER_ROUND - len(keep_idx)
        print(f'  round {rnd+1}: kept {len(keep_idx)}/{CLIENTS_PER_ROUND} '
              f'({n_excluded} excluded)')

    metrics = evaluate(global_model, X_test, y_test)
    return metrics, reputation, mal_ids

## Table 3 — Attack-percentage sweep (BiAB-IoT + FedAvg reference)

We run five attack conditions (0/5/10/15/20 %) for standard FedAvg (B2,
`use_P1=False`, `use_P2=False`) and for the full BiAB-IoT (`use_P1=True`,
`use_P2=True`).

Standard FedAvg is included here because it's the reference point for
Table 3 and it's cheap to re-run with the same config. B1, B3, B4 numbers
for Table 3 come from separate specialised runs — omitted from this
notebook for runtime, added to future work.

In [ ]:
from time import time

ATTACK_PCTS = [0.00, 0.05, 0.10, 0.15, 0.20]

for pct in ATTACK_PCTS:
    print(f'\n=== FedAvg (B2) @ {int(pct*100)}% malicious ===')
    t0 = time()
    m, _, _ = run_biab_iot(attack_pct=pct, use_P1=False, use_P2=False)
    print(f'  time: {time()-t0:.1f}s  metrics: {m}')
    save_result(f'Table3_B2_FedAvg_{int(pct*100):02d}pct', m,
                {'attack_pct': pct, 'method': 'FedAvg'})

for pct in ATTACK_PCTS:
    print(f'\n=== BiAB-IoT (P1+P2) @ {int(pct*100)}% malicious ===')
    t0 = time()
    m, _, _ = run_biab_iot(attack_pct=pct, use_P1=True, use_P2=True)
    print(f'  time: {time()-t0:.1f}s  metrics: {m}')
    save_result(f'Table3_BiAB_IoT_{int(pct*100):02d}pct', m,
                {'attack_pct': pct, 'method': 'BiAB-IoT'})

## Table 4 — Ablation (10 % malicious, fixed condition)

Three ablation configurations at the standard 10 % attack condition:

* **P1 only** — exclusion via dual-condition trigger, no reputation weighting
  of the retained clients.
* **P2 only** — reputation-weighted aggregation over all clients, no
  exclusion.
* **P1 + P2** — the full BiAB-IoT (already in the Table 3 results at 10 %,
  re-run here for a clean ablation record).

In [ ]:
ATTACK_PCT_ABL = 0.10

for name, use_P1, use_P2 in [
    ('P1_only',  True,  False),
    ('P2_only',  False, True),
    ('P1_plus_P2', True, True),
]:
    print(f'\n=== Ablation: {name} @ {int(ATTACK_PCT_ABL*100)}% ===')
    t0 = time()
    m, _, _ = run_biab_iot(attack_pct=ATTACK_PCT_ABL, use_P1=use_P1, use_P2=use_P2)
    print(f'  time: {time()-t0:.1f}s  metrics: {m}')
    save_result(f'Table4_ablation_{name}', m,
                {'attack_pct': ATTACK_PCT_ABL, 'use_P1': use_P1, 'use_P2': use_P2})

## Summary

Result files persisted for Notebook D to aggregate:

* `Table3_B2_FedAvg_00pct.pkl` ... `Table3_B2_FedAvg_20pct.pkl`
* `Table3_BiAB_IoT_00pct.pkl` ... `Table3_BiAB_IoT_20pct.pkl`
* `Table4_ablation_P1_only.pkl`, `Table4_ablation_P2_only.pkl`,
  `Table4_ablation_P1_plus_P2.pkl`
